# Customer Churn Prediction — Step 6: Machine Learning Model Training

**Input:** `data/preprocessed_splits.pkl` — train/test splits and unfitted `ColumnTransformer` from Step 5  
**Output:** `data/trained_models.pkl` — dictionary of four fully trained scikit-learn `Pipeline` objects  

### What this notebook does
1. Loads the train/test splits and the unfitted preprocessing pipeline produced in Step 5.
2. Constructs **four scikit-learn `Pipeline` objects**, each consisting of the shared `ColumnTransformer` preprocessor followed by a classifier.
3. Trains every pipeline **exclusively on `X_train` / `y_train`**.
4. Confirms that all four models fitted successfully.
5. Reports a brief training-set accuracy check as a sanity signal only — **not** as a performance evaluation.
6. Serialises the trained pipelines to disk for use in Step 7 (Evaluation).

### What this notebook does NOT do
- Does **not** use `X_test` or `y_test` for training, model selection, or hyperparameter tuning.
- Does **not** perform extensive hyperparameter search (that is Step 8 — Hyperparameter Tuning).
- Does **not** declare a best model — that decision belongs in Step 7 after full evaluation.
- Does **not** perform EDA or re-run preprocessing outside of the pipeline.
- Does **not** modify the cleaned CSV.

---

## 1. Setup — Import Libraries

We import the four classifier classes from scikit-learn along with `Pipeline`, `joblib`, and supporting utilities.  
All random operations use `RANDOM_STATE = 42` for full reproducibility.

In [1]:
import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# ── Resolve paths ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR   = os.path.abspath('')
PROJECT_ROOT   = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == 'notebook' else NOTEBOOK_DIR
SPLITS_PATH    = os.path.join(PROJECT_ROOT, 'data', 'preprocessed_splits.pkl')
CLEAN_PATH     = os.path.join(PROJECT_ROOT, 'data', 'telco_churn_cleaned.csv')
MODELS_PATH    = os.path.join(PROJECT_ROOT, 'data', 'trained_models.pkl')

print('Splits file  :', SPLITS_PATH)
print('Clean CSV    :', CLEAN_PATH)
print('Models output:', MODELS_PATH)

Splits file  : c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\preprocessed_splits.pkl
Clean CSV    : c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\telco_churn_cleaned.csv
Models output: c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\trained_models.pkl


---
## 2. Load Train / Test Splits and Preprocessor

We load the artifacts serialised in Step 5: the raw (unencoded) `X_train`, `X_test`, `y_train`, `y_test` DataFrames and the **unfitted** `ColumnTransformer`.  

If the `pkl` file is not available (e.g., Step 5 has not been run yet), the cell automatically rebuilds the splits from the cleaned CSV using the **identical parameters** (`test_size=0.20`, `random_state=42`, `stratify=y`) so the notebook is always runnable in isolation without compromising reproducibility.

In [2]:
def _build_preprocessor(numerical_features, categorical_features):
    """Reconstruct the unfitted ColumnTransformer — identical to Step 5."""
    numerical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer(
        transformers=[
            ('num', numerical_pipeline,   numerical_features),
            ('cat', categorical_pipeline, categorical_features),
        ],
        remainder='drop'
    )


if os.path.exists(SPLITS_PATH):
    print(f'Loading splits from {SPLITS_PATH} ...')
    artifacts = joblib.load(SPLITS_PATH)
    X_train             = artifacts['X_train']
    X_test              = artifacts['X_test']
    y_train             = artifacts['y_train']
    y_test              = artifacts['y_test']
    numerical_features  = artifacts['numerical_features']
    categorical_features = artifacts['categorical_features']
    # Always reconstruct a fresh unfitted preprocessor (the pkl version may have
    # been fitted during the Step 5 preview and should not be reused)
    preprocessor = _build_preprocessor(numerical_features, categorical_features)
    print('Loaded from pkl.')
else:
    print(f'{SPLITS_PATH} not found — rebuilding from cleaned CSV ...')
    df = pd.read_csv(CLEAN_PATH)
    X  = df.drop(columns=['Churn'])
    y  = (df['Churn'] == 'Yes').astype(int)
    numerical_features   = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
    categorical_features = [c for c in X.columns if c not in numerical_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    preprocessor = _build_preprocessor(numerical_features, categorical_features)
    print('Rebuilt from cleaned CSV.')

print()
print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}  |  class dist: {y_train.value_counts().sort_index().to_dict()}')
print(f'y_test  : {y_test.shape}   |  class dist: {y_test.value_counts().sort_index().to_dict()}')
print(f'Numerical features  : {numerical_features}')
print(f'Categorical features: {categorical_features}')
print(f'Preprocessor fitted : {hasattr(preprocessor, "transformers_")}')

Loading splits from c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\preprocessed_splits.pkl ...
Loaded from pkl.

X_train : (5616, 19)
X_test  : (1405, 19)
y_train : (5616,)  |  class dist: {0: 4131, 1: 1485}
y_test  : (1405,)   |  class dist: {0: 1033, 1: 372}
Numerical features  : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Preprocessor fitted : False


---
## 3. Define the Four Model Pipelines

Each model is wrapped in a scikit-learn `Pipeline` with exactly two steps:

```
Pipeline([
    ('preprocessor', ColumnTransformer(...)),   # shared, unfitted
    ('classifier',   SomeClassifier(...))       # model-specific
])
```

Using the **same unfitted `ColumnTransformer`** as the first step of every pipeline ensures:  
- All models see identically transformed features.  
- The scaler and encoder are fitted independently within each pipeline's `.fit()` call — no cross-contamination between pipelines.  
- In Step 7, calling `pipeline.predict(X_test)` automatically applies the correct training-fitted transformation before prediction.  

### Hyperparameter choices

These are **baseline configurations** — sensible defaults intended to give a fair first comparison without tuning:

| Model | Key parameters | Rationale |
|---|---|---|
| Logistic Regression | `C=1.0`, `max_iter=1000`, `class_weight='balanced'` | `max_iter=1000` ensures convergence on 45 features; `class_weight='balanced'` compensates for the 73.6/26.4 class imbalance by up-weighting the minority (churned) class during optimisation — appropriate here because failing to identify a churning customer (false negative) is typically more costly than a false alarm |
| Decision Tree | `max_depth=10`, `min_samples_leaf=20`, `class_weight='balanced'` | Unrestricted depth causes severe overfitting on tabular data; `max_depth=10` and `min_samples_leaf=20` provide regularisation; `class_weight='balanced'` for same reason as LR |
| Random Forest | `n_estimators=200`, `max_depth=None`, `min_samples_leaf=5`, `class_weight='balanced_subsample'` | 200 trees gives stable estimates; `balanced_subsample` rebalances each bootstrap sample independently — the standard approach for imbalanced classification in Random Forests |
| Gradient Boosting | `n_estimators=200`, `learning_rate=0.1`, `max_depth=4`, `subsample=0.8` | Standard conservative baseline; `subsample=0.8` (stochastic GB) reduces overfitting; GradientBoostingClassifier does not support `class_weight`, so we rely on the natural boosting focus on misclassified examples |

In [3]:
import copy

# Each pipeline gets its own deep copy of the unfitted preprocessor
# so that fitting one pipeline does not affect the others.

pipelines = {

    'Logistic Regression': Pipeline([
        ('preprocessor', copy.deepcopy(preprocessor)),
        ('classifier', LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            solver='lbfgs'
        ))
    ]),

    'Decision Tree': Pipeline([
        ('preprocessor', copy.deepcopy(preprocessor)),
        ('classifier', DecisionTreeClassifier(
            max_depth=10,
            min_samples_leaf=20,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', copy.deepcopy(preprocessor)),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=5,
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    'Gradient Boosting': Pipeline([
        ('preprocessor', copy.deepcopy(preprocessor)),
        ('classifier', GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            random_state=RANDOM_STATE
        ))
    ]),

}

print(f'Pipelines defined: {list(pipelines.keys())}')
print()
for name, pipe in pipelines.items():
    clf = pipe.named_steps['classifier']
    print(f'{name}:')
    print(f'  {clf}')
    print()

Pipelines defined: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting']

Logistic Regression:
  LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

Decision Tree:
  DecisionTreeClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=20, random_state=42)

Random Forest:
  RandomForestClassifier(class_weight='balanced_subsample', min_samples_leaf=5,
                       n_estimators=200, n_jobs=-1, random_state=42)

Gradient Boosting:
  GradientBoostingClassifier(max_depth=4, n_estimators=200, random_state=42,
                           subsample=0.8)



---
## 4. Train All Models

Each pipeline is trained by calling `.fit(X_train, y_train)`.  
This single call:
1. **Fits** the `ColumnTransformer` on `X_train` — learning the scaler's mean/std and the encoder's category vocabulary exclusively from training data.
2. **Transforms** `X_train` using those fitted parameters.
3. **Fits** the classifier on the transformed training features.

The test set `X_test` / `y_test` is never touched in this section.  
Training time is recorded for each model as a practical reference.

In [4]:
trained_models = {}
training_times = {}

print('Training models on X_train / y_train ...')
print(f'  Training set: {X_train.shape[0]:,} rows × {X_train.shape[1]} raw features')
print()

for name, pipeline in pipelines.items():
    t0 = time.perf_counter()
    pipeline.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0
    trained_models[name] = pipeline
    training_times[name] = elapsed
    print(f'  [{name:<22}]  fitted in {elapsed:>6.2f}s')

print()
print('All models trained.')

Training models on X_train / y_train ...
  Training set: 5,616 rows × 19 raw features

  [Logistic Regression   ]  fitted in   0.08s
  [Decision Tree         ]  fitted in   0.06s
  [Random Forest         ]  fitted in   0.51s
  [Gradient Boosting     ]  fitted in   2.16s

All models trained.


---
## 5. Confirm Successful Fitting

We verify that every pipeline's preprocessor and classifier have been fitted by checking for the presence of fit-time attributes:  
- `ColumnTransformer` gains `transformers_` (list of fitted transformers) after fitting.  
- `LogisticRegression` gains `coef_` after fitting.  
- `DecisionTreeClassifier` and `RandomForestClassifier` gain `n_features_in_` after fitting.  
- `GradientBoostingClassifier` gains `estimators_` after fitting.

In [5]:
fit_checks = {
    'Logistic Regression': lambda p: hasattr(p.named_steps['classifier'], 'coef_'),
    'Decision Tree':       lambda p: hasattr(p.named_steps['classifier'], 'n_features_in_'),
    'Random Forest':       lambda p: hasattr(p.named_steps['classifier'], 'n_features_in_'),
    'Gradient Boosting':   lambda p: hasattr(p.named_steps['classifier'], 'estimators_'),
}

print('=== Fitting Confirmation ===')
all_ok = True
for name, pipeline in trained_models.items():
    preprocessor_fitted = hasattr(pipeline.named_steps['preprocessor'], 'transformers_')
    classifier_fitted   = fit_checks[name](pipeline)
    status = 'OK' if (preprocessor_fitted and classifier_fitted) else 'FAIL'
    if status == 'FAIL':
        all_ok = False
    print(f'  {name:<22}  preprocessor_fitted={preprocessor_fitted}  classifier_fitted={classifier_fitted}  [{status}]')

print()
print('All pipelines successfully fitted.' if all_ok else 'WARNING: one or more pipelines did not fit correctly.')

=== Fitting Confirmation ===
  Logistic Regression     preprocessor_fitted=True  classifier_fitted=True  [OK]
  Decision Tree           preprocessor_fitted=True  classifier_fitted=True  [OK]
  Random Forest           preprocessor_fitted=True  classifier_fitted=True  [OK]
  Gradient Boosting       preprocessor_fitted=True  classifier_fitted=True  [OK]

All pipelines successfully fitted.


---
## 6. Preprocessed Feature Space (Reference)

We record the full list of feature names produced by the `ColumnTransformer` after fitting.  
This is purely informational — it confirms the 45 output columns (4 scaled numerical + 41 one-hot encoded) and names each OHE column, which is useful when inspecting feature importances in Step 7.

In [6]:
# Extract feature names from any one of the fitted pipelines
fitted_ct = trained_models['Random Forest'].named_steps['preprocessor']

num_names = numerical_features
ohe       = fitted_ct.named_transformers_['cat'].named_steps['encoder']
ohe_names = ohe.get_feature_names_out(categorical_features).tolist()
all_feature_names = num_names + ohe_names

print(f'Total preprocessed features: {len(all_feature_names)}')
print(f'  Numerical (scaled) : {len(num_names)}')
print(f'  One-hot encoded    : {len(ohe_names)}')
print()
print('All feature names:')
for i, name in enumerate(all_feature_names):
    print(f'  {i+1:>2}. {name}')

Total preprocessed features: 45
  Numerical (scaled) : 4
  One-hot encoded    : 41

All feature names:
   1. SeniorCitizen
   2. tenure
   3. MonthlyCharges
   4. TotalCharges
   5. gender_Female
   6. gender_Male
   7. Partner_No
   8. Partner_Yes
   9. Dependents_No
  10. Dependents_Yes
  11. PhoneService_No
  12. PhoneService_Yes
  13. MultipleLines_No
  14. MultipleLines_No phone service
  15. MultipleLines_Yes
  16. InternetService_DSL
  17. InternetService_Fiber optic
  18. InternetService_No
  19. OnlineSecurity_No
  20. OnlineSecurity_No internet service
  21. OnlineSecurity_Yes
  22. OnlineBackup_No
  23. OnlineBackup_No internet service
  24. OnlineBackup_Yes
  25. DeviceProtection_No
  26. DeviceProtection_No internet service
  27. DeviceProtection_Yes
  28. TechSupport_No
  29. TechSupport_No internet service
  30. TechSupport_Yes
  31. StreamingTV_No
  32. StreamingTV_No internet service
  33. StreamingTV_Yes
  34. StreamingMovies_No
  35. StreamingMovies_No internet servi

---
## 7. Training-Set Accuracy — Sanity Check Only

We compute training-set accuracy for each model as a **sanity check** — not as a performance metric.  

What training accuracy tells us:  
- If a model achieves very low training accuracy (e.g., < 60%), something is likely wrong with the pipeline.  
- If a model achieves near-100% training accuracy (e.g., Decision Tree without depth restriction), it is likely memorising the training set.  

What training accuracy does **not** tell us:  
- It says nothing about generalisation to unseen data.  
- It should not be used to compare models or select a winner.  

The full evaluation on `X_test` — using accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrices — is the subject of Step 7.

In [7]:
print('=== Training-Set Accuracy (sanity check only — not a performance evaluation) ===')
print(f'{"Model":<25} {"Train Accuracy":>16} {"Train Time":>12}')
print('-' * 56)
for name, pipeline in trained_models.items():
    y_train_pred = pipeline.predict(X_train)
    train_acc    = accuracy_score(y_train, y_train_pred)
    t_sec        = training_times[name]
    print(f'{name:<25} {train_acc:>15.4f}  {t_sec:>10.2f}s')
print()
print('NOTE: Training accuracy is not used for model selection.')
print('Full evaluation (precision / recall / F1 / ROC-AUC) will be performed in Step 7.')

=== Training-Set Accuracy (sanity check only — not a performance evaluation) ===
Model                       Train Accuracy   Train Time
--------------------------------------------------------
Logistic Regression                0.7514        0.08s
Decision Tree                      0.7901        0.06s
Random Forest                      0.8446        0.51s
Gradient Boosting                  0.8805        2.16s

NOTE: Training accuracy is not used for model selection.
Full evaluation (precision / recall / F1 / ROC-AUC) will be performed in Step 7.


**Expected observations:**  
- **Logistic Regression** will show moderate training accuracy — it is a linear model and cannot capture non-linear patterns in the training set.  
- **Decision Tree** (depth=10) will show higher training accuracy than LR — trees can fit more complex patterns.  
- **Random Forest** and **Gradient Boosting** will generally show the highest training accuracy — ensembles of many trees can fit the training data very well.  
- A large gap between training accuracy and (future) test accuracy would signal overfitting. That analysis belongs in Step 7.

---
## 8. Save Trained Pipelines

All four trained pipelines are saved to `data/trained_models.pkl` using `joblib`.  
Step 7 (Model Evaluation) will load this file directly, ensuring it evaluates the exact same fitted objects produced here — no re-training, no risk of different random initialisations.

We also save the `all_feature_names` list, which Step 7 needs for feature importance plots.

In [8]:
model_artifacts = {
    'trained_models':    trained_models,      # dict: name -> fitted Pipeline
    'training_times':    training_times,      # dict: name -> seconds
    'all_feature_names': all_feature_names,   # list of 45 preprocessed feature names
    'numerical_features':   numerical_features,
    'categorical_features': categorical_features,
    'random_state': RANDOM_STATE,
}

joblib.dump(model_artifacts, MODELS_PATH)
print(f'Trained models saved to: {MODELS_PATH}')

# Reload and verify
loaded_models = joblib.load(MODELS_PATH)
print()
print('Verification — keys in saved file:', list(loaded_models.keys()))
print('Models saved:', list(loaded_models['trained_models'].keys()))
for name, pipe in loaded_models['trained_models'].items():
    pre_fitted = hasattr(pipe.named_steps['preprocessor'], 'transformers_')
    print(f'  {name:<22}  preprocessor_fitted={pre_fitted}')

Trained models saved to: c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\trained_models.pkl

Verification — keys in saved file: ['trained_models', 'training_times', 'all_feature_names', 'numerical_features', 'categorical_features', 'random_state']
Models saved: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting']
  Logistic Regression     preprocessor_fitted=True
  Decision Tree           preprocessor_fitted=True
  Random Forest           preprocessor_fitted=True
  Gradient Boosting       preprocessor_fitted=True


---
## 9. Step Summary

A consolidated record of what was done in this notebook.

In [9]:
print('=' * 62)
print('STEP 6 — MODEL TRAINING SUMMARY')
print('=' * 62)
print()
print(f'Training set          : {X_train.shape[0]:,} rows × {X_train.shape[1]} raw features')
print(f'Test set (held out)   : {X_test.shape[0]:,} rows × {X_test.shape[1]} raw features')
print(f'Preprocessed features : {len(all_feature_names)} (4 scaled + 41 OHE)')
print()
print('Models trained:')
for name, pipeline in trained_models.items():
    clf = pipeline.named_steps['classifier']
    print(f'  {name:<22}  {clf.__class__.__name__}')
print()
print(f'Random state          : {RANDOM_STATE}')
print(f'Test set used for     : evaluation only (Step 7)')
print(f'Artifacts saved to    : {MODELS_PATH}')
print('=' * 62)

STEP 6 — MODEL TRAINING SUMMARY

Training set          : 5,616 rows × 19 raw features
Test set (held out)   : 1,405 rows × 19 raw features
Preprocessed features : 45 (4 scaled + 41 OHE)

Models trained:
  Logistic Regression     LogisticRegression
  Decision Tree           DecisionTreeClassifier
  Random Forest           RandomForestClassifier
  Gradient Boosting       GradientBoostingClassifier

Random state          : 42
Test set used for     : evaluation only (Step 7)
Artifacts saved to    : c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\trained_models.pkl


---
## 10. Model Training Decisions

### Why multiple models are compared
No single algorithm dominates on all tabular classification tasks. Different models make different inductive assumptions:
- **Logistic Regression** assumes a linear decision boundary in the (transformed) feature space. If the true relationship between features and churn is approximately linear after encoding, it will perform well and produce highly interpretable coefficients.
- **Decision Tree** makes no linearity assumption and can capture non-linear interactions, but a single tree is susceptible to overfitting.
- **Random Forest** and **Gradient Boosting** are ensemble methods that combine many trees to reduce variance (RF) or bias (GB). They consistently perform well on structured tabular data.

Training multiple models and comparing them on a held-out test set (Step 7) gives an evidence-based foundation for model selection — rather than assuming one approach is superior.

---

### Why Logistic Regression is useful as a baseline
Logistic Regression is the standard linear baseline for binary classification.  
- It is **fast**, **interpretable** (the sign and magnitude of each coefficient reflect that feature's contribution to the log-odds of churn), and well-understood theoretically.
- Its performance represents a floor: if a more complex model (Random Forest, Gradient Boosting) cannot substantially outperform it, that is meaningful information — it suggests the relationship between features and churn may be largely linear, or that the additional complexity is not justified.
- Regularisation (`C=1.0`) prevents overfitting to the training set.
- `class_weight='balanced'` adjusts the loss function so that misclassifying a churned customer (the minority, more commercially costly class) is penalised proportionally more — appropriate given the 73.6 / 26.4 imbalance in this dataset.

---

### Why tree-based models are included
The EDA in Step 4 revealed several non-linear patterns — particularly the sharp drop in churn rate with increasing tenure, and interactions between contract type, internet service, and add-on services.  
Tree-based models handle these patterns naturally:
- They do not require the relationship to be linear.
- They automatically discover interactions between features (e.g., a split on `Contract == Month-to-month` followed by a split on `tenure < 12` captures the highest-risk segment).
- Ensemble methods (Random Forest, Gradient Boosting) are among the most consistently high-performing algorithms on structured tabular data in practice.

---

### Why the same preprocessing pipeline is used for all models
Using a single shared `ColumnTransformer` definition (with independent deep copies per pipeline) guarantees:
- All four models receive **identically encoded and scaled features** — the comparison is fair.
- There is no risk of accidentally using raw string columns for one model and encoded columns for another.
- The preprocessing is encapsulated inside each `Pipeline`, so calling `pipeline.predict(new_data)` always applies the correct transformation — the model is self-contained and safe to deploy.

Note: `StandardScaler` is applied to all models including tree-based ones. Scaling does not hurt trees (they are invariant to monotone transformations), and it avoids maintaining separate pipeline variants.

---

### Why the test set is kept entirely separate
The test set `X_test` / `y_test` represents data the model has never seen.  
Its purpose is to simulate how the model would perform on real future customers arriving after the model is deployed.  

If the test set were used to make any training decision — selecting hyperparameters, choosing which model to use, or deciding when to stop training — then the reported test metrics would be optimistically biased. The model would have been inadvertently tuned to that specific test partition and the reported performance would not generalise.

The correct flow is:
- **Train** on `X_train` / `y_train` only.
- **Select hyperparameters** using cross-validation on `X_train` only (Step 8).
- **Evaluate** the final, fixed model on `X_test` exactly once (Step 7).

By keeping `X_test` untouched until Step 7, the evaluation metrics will honestly reflect expected performance on unseen data.